# 06 — Gold Dimension Tables
## Silver → dim_customer, dim_product, dim_region

**Purpose:** Build the three dimension tables of the Gold star schema.
Each dimension is the authoritative record of one business entity.
No measures — purely descriptive attributes for joining and filtering.

**Input :** `sales_silver.clean_superstore`
**Outputs:**
  - `sales_gold.dim_customer`
  - `sales_gold.dim_product`
  - `sales_gold.dim_region`
**Layer  :** Gold (dimensions)


## 1. Load config

In [0]:
%run ./00_setup_and_config

In [0]:
log("INFO", "06_gold_dim_tables", "Starting Gold dimension table build")


## 2. Read Silver - single source for all three dimensions

In [0]:
# cache() is NOT supported on Databricks Serverless.
# Databricks Serverless manages IO caching internally via the
# Delta cache — manual persist is unavailable and unnecessary.
#
# For this dataset size (9,994 rows), reading Silver three times
# (once per dimension) is fast. Each read hits Delta's internal
# file cache after the first scan.
#
# In a self-managed cluster with a very large Silver table,
# you would use .cache() here to avoid three full table scans.

df_silver = spark.table(SILVER_TABLE)

row_count = df_silver.count()

log("INFO", "06_gold_dim_tables", f"Silver loaded: {row_count:,} rows")
print(f"  Silver rows      : {row_count:,}")
print(f"  Will be used for : dim_customer, dim_product, dim_region")
print(f"  ✅ Silver table ready")
print(f"  ℹ️  Note: manual caching not available on Serverless —")
print(f"     Delta IO cache handles optimisation automatically")


## 3. Build dim_customer

In [0]:
# WHAT it represents: every unique customer in the dataset
# GRAIN            : one row per customer_id
# ATTRIBUTES       : who they are and what segment they belong to
#
# Design decisions documented:
#
#   customer_id    → natural key (comes from source system)
#   customer_name  → display name for reports
#   segment        → Consumer / Corporate / Home Office
#                    This is a customer-level attribute — a customer
#                    belongs to one segment consistently across all orders
#
# What we deliberately EXCLUDE:
#   city, state, region → these are order-level attributes, not customer
#                         attributes. A customer might order from New York
#                         one month and Chicago the next. Putting geography
#                         in dim_customer would be wrong — it belongs in
#                         fact_sales as a degenerate dimension.
#
# This is a core dimensional modelling decision. Getting it wrong
# causes incorrect aggregations at query time.

from pyspark.sql import functions as F

df_customer = (
    df_silver
    .select("customer_id", "customer_name", "segment")
    .distinct()
    .orderBy("customer_id")
)

# Add surrogate key
df_customer = df_customer.withColumn(
    "customer_key",
    F.monotonically_increasing_id()
)

# Add dimension metadata
df_customer = df_customer.withColumns({
    "_dim_created_at"  : F.current_timestamp(),
    "_dim_version"     : F.lit("1.0"),
    "_is_active"       : F.lit(True),
})

# Reorder columns — key first, then natural key, then attributes
df_customer = df_customer.select(
    "customer_key",
    "customer_id",
    "customer_name",
    "segment",
    "_dim_created_at",
    "_dim_version",
    "_is_active",
)

customer_count = df_customer.count()

print("=== dim_customer PREVIEW ===\n")
df_customer.show(10, truncate=False)
print(f"  Unique customers : {customer_count:,}")

# Verify uniqueness at customer_id grain
distinct_ids = df_customer.select("customer_id").distinct().count()
if customer_count == distinct_ids:
    print(f"  ✅ customer_id is unique — grain verified")
else:
    raise Exception(f"dim_customer grain violated: {customer_count} rows, {distinct_ids} distinct IDs")

log("INFO", "06_gold_dim_tables", f"dim_customer built: {customer_count} customers")

In [0]:
(
    df_customer
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_CUSTOMER)
)

log("INFO", "06_gold_dim_tables", f"dim_customer written: {DIM_CUSTOMER}")
print(f"  ✅ {DIM_CUSTOMER} written — {customer_count:,} rows")


## 4. Build dim_product

In [0]:
# WHAT it represents: every unique product in the dataset
# GRAIN            : one row per product_id
# ATTRIBUTES       : product_name, category, sub_category
#
# DATA QUALITY ISSUE DISCOVERED AND RESOLVED:
#   32 product_ids have multiple name variants in the source data.
#   Example: same product_id recorded with slightly different names
#   across different orders (colour variants, formatting differences).
#   Category and sub_category are consistent — only names conflict.
#
# RESOLUTION STRATEGY: take the most frequently occurring name
#   per product_id. This is the "majority vote" approach —
#   the name that appears most often across orders is the
#   canonical name the source system uses most consistently.
#
#   Alternative strategies (documented for interviews):
#     - Take the longest name  → captures most detail
#     - Take the most recent   → requires order_date join
#     - Flag for manual review → correct in large enterprise pipelines
#   We chose most frequent because it is deterministic, reproducible,
#   and defensible to business stakeholders.

from pyspark.sql import functions as F
from pyspark.sql import Window

# Step 1: Count how often each product_id + product_name combo appears
# across all transactions — the frequency is our tiebreaker
df_product_freq = (
    df_silver
    .groupBy("product_id", "product_name", "category", "sub_category")
    .agg(F.count("*").alias("occurrence_count"))
)

# Step 2: Rank names within each product_id by frequency
# rank=1 means this name appears most often for this product_id
window_spec = Window.partitionBy("product_id").orderBy(
    F.col("occurrence_count").desc(),
    F.col("product_name").asc()   # alphabetical tiebreaker for determinism
)

df_product_ranked = df_product_freq.withColumn(
    "name_rank",
    F.rank().over(window_spec)
)

# Step 3: Keep only rank 1 — the most frequent name per product_id
df_product = (
    df_product_ranked
    .filter(F.col("name_rank") == 1)
    .select("product_id", "product_name", "category", "sub_category")
    .orderBy("category", "sub_category", "product_id")
)

# Step 4: Add surrogate key
df_product = df_product.withColumn(
    "product_key",
    F.monotonically_increasing_id()
)

# Step 5: Enrich with pricing context from Silver
df_price = (
    df_silver
    .groupBy("product_id")
    .agg(
        F.round(F.avg("unit_price"), 2).alias("avg_unit_price"),
        F.round(F.min("unit_price"), 2).alias("min_unit_price"),
        F.round(F.max("unit_price"), 2).alias("max_unit_price"),
    )
)

df_product = df_product.join(df_price, on="product_id", how="left")

# Step 6: Add dimension metadata
df_product = df_product.withColumns({
    "_dim_created_at"   : F.current_timestamp(),
    "_dim_version"      : F.lit("1.0"),
    "_is_active"        : F.lit(True),
    "_name_resolution"  : F.lit("most_frequent"),
})

# Step 7: Final column order
df_product = df_product.select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "sub_category",
    "avg_unit_price",
    "min_unit_price",
    "max_unit_price",
    "_dim_created_at",
    "_dim_version",
    "_is_active",
    "_name_resolution",
)

product_count = df_product.count()

print("=== dim_product PREVIEW ===\n")
df_product.show(10, truncate=False)
print(f"  Unique products  : {product_count:,}")

# Verify grain is now clean
distinct_product_ids = df_product.select("product_id").distinct().count()
if product_count == distinct_product_ids:
    print(f"  ✅ product_id is unique — grain verified after resolution")
else:
    raise Exception(
        f"Grain still violated after resolution: "
        f"{product_count} rows, {distinct_product_ids} distinct IDs"
    )

# Show category distribution
print("\n  Products per category:\n")
df_product.groupBy("category") \
    .agg(
        F.count("*").alias("product_count"),
        F.countDistinct("sub_category").alias("sub_categories"),
        F.round(F.avg("avg_unit_price"), 2).alias("avg_price"),
    ) \
    .orderBy("category") \
    .show()

log("INFO", "06_gold_dim_tables",
    f"dim_product built: {product_count} products "
    f"(32 name conflicts resolved via majority vote)")

In [0]:
(
    df_product
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_PRODUCT)
)

log("INFO", "06_gold_dim_tables", f"dim_product written: {DIM_PRODUCT}")
print(f"  ✅ {DIM_PRODUCT} written — {product_count:,} rows")


## 5. Build dim_region

In [0]:
# WHAT it represents: every unique geographic entity in the dataset
# GRAIN            : one row per region + state + city combination
#
# Design decision — what is the grain of dim_region?
#
# Option A: one row per region (4 rows — East, West, Central, South)
#   Too coarse — analysts can't drill into state or city level
#
# Option B: one row per city (many rows — most granular)
#   Better — preserves full geographic hierarchy
#
# Option C: one row per region + state + city (chosen)
#   Best — captures the full hierarchy in one denormalised dimension
#   Allows GROUP BY at any level: region, state, or city
#   Consistent with Kimball dimensional modelling principles
#
# The grain is: one row per unique region + state + city combination

df_region = (
    df_silver
    .select("region", "state", "city", "country", "postal_code")
    .distinct()
    .orderBy("region", "state", "city")
)

# Add surrogate key
df_region = df_region.withColumn(
    "region_key",
    F.monotonically_increasing_id()
)

# Enrich with aggregated sales context per region
# This gives analysts a quick reference for region size
df_region_stats = (
    df_silver
    .groupBy("region")
    .agg(
        F.countDistinct("customer_id").alias("customer_count"),
        F.countDistinct("order_id").alias("order_count"),
        F.round(F.sum("sales"), 2).alias("total_sales"),
    )
)

df_region = df_region.join(df_region_stats, on="region", how="left")

# Add dimension metadata
df_region = df_region.withColumns({
    "_dim_created_at" : F.current_timestamp(),
    "_dim_version"    : F.lit("1.0"),
})

# Final column order
df_region = df_region.select(
    "region_key",
    "region",
    "state",
    "city",
    "country",
    "postal_code",
    "customer_count",
    "order_count",
    "total_sales",
    "_dim_created_at",
    "_dim_version",
)

region_count = df_region.count()

print("=== dim_region PREVIEW ===\n")
df_region.show(10, truncate=False)
print(f"  Unique region+state+city combinations : {region_count:,}")

# Show region summary
print("\n  Geographic summary:\n")
df_region.groupBy("region") \
    .agg(
        F.countDistinct("state").alias("states"),
        F.countDistinct("city").alias("cities"),
        F.round(F.sum("total_sales"), 2).alias("region_sales"),
    ) \
    .orderBy("region_sales", ascending=False) \
    .show()

log("INFO", "06_gold_dim_tables", f"dim_region built: {region_count} geographic combos")

In [0]:
(
    df_region
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIM_REGION)
)

log("INFO", "06_gold_dim_tables", f"dim_region written: {DIM_REGION}")
print(f"  ✅ {DIM_REGION} written — {region_count:,} rows")


## 6. Cache management note

In [0]:
# On Databricks Serverless, manual cache/unpersist is not available.
# Databricks manages memory and IO caching automatically.
# No action needed here.

print("  ✅ No manual cache to release on Serverless")
print("  ℹ️  Delta IO cache is managed automatically by the platform")
log("INFO", "06_gold_dim_tables", "Cache management: handled by Serverless platform")


## 7. Validate the star schema - referential integrity checks

In [0]:
# In a relational database, foreign key constraints enforce referential
# integrity automatically. Delta Lake does not enforce foreign keys.
# We validate manually — checking that every foreign key in fact_sales
# has a matching record in the dimension table it references.
#
# If customer_id in fact_sales has no match in dim_customer, that
# transaction becomes an "orphan fact" — it exists but can never be
# described. Analysts would see NULL customer names in reports.

from pyspark.sql import functions as F

df_fact    = spark.table(FACT_TABLE)
df_dim_cust = spark.table(DIM_CUSTOMER)
df_dim_prod = spark.table(DIM_PRODUCT)
df_dim_reg  = spark.table(DIM_REGION)

print("=== STAR SCHEMA REFERENTIAL INTEGRITY ===\n")

# Check 1: Every customer_id in fact_sales exists in dim_customer
orphan_customers = (
    df_fact
    .join(df_dim_cust, on="customer_id", how="left_anti")
    .count()
)
status = "✅" if orphan_customers == 0 else f"⚠️  {orphan_customers} orphans"
print(f"  fact → dim_customer  : {status}")

# Check 2: Every product_id in fact_sales exists in dim_product
orphan_products = (
    df_fact
    .join(df_dim_prod, on="product_id", how="left_anti")
    .count()
)
status = "✅" if orphan_products == 0 else f"⚠️  {orphan_products} orphans"
print(f"  fact → dim_product   : {status}")

# Check 3: Every region in fact_sales exists in dim_region
orphan_regions = (
    df_fact
    .join(df_dim_reg, on="region", how="left_anti")
    .count()
)
status = "✅" if orphan_regions == 0 else f"⚠️  {orphan_regions} orphans"
print(f"  fact → dim_region    : {status}")

total_orphans = orphan_customers + orphan_products + orphan_regions
if total_orphans == 0:
    print(f"\n  ✅ Star schema referential integrity confirmed")
    print(f"     All fact rows have matching dimension records")
else:
    log("WARN", "06_gold_dim_tables",
        f"Referential integrity issues: {total_orphans} orphan facts found")

log("INFO", "06_gold_dim_tables", "Star schema validation complete")


## 8. The star schema working end-to-end

In [0]:
%sql
-- Star schema join — the gold standard query pattern
-- Every dimension enriching the fact with its descriptive context

SELECT
    c.customer_name,
    c.segment,
    p.category,
    p.sub_category,
    r.region,
    r.state,
    f.order_year,
    f.order_quarter,
    COUNT(DISTINCT f.order_id)          AS orders,
    SUM(f.quantity)                     AS units_sold,
    ROUND(SUM(f.sales), 2)              AS total_sales,
    ROUND(SUM(f.profit), 2)             AS total_profit,
    ROUND(AVG(f.profit_margin) * 100, 2) AS avg_margin_pct
FROM sales_gold.fact_sales     f
JOIN sales_gold.dim_customer   c ON f.customer_id = c.customer_id
JOIN sales_gold.dim_product    p ON f.product_id  = p.product_id
JOIN sales_gold.dim_region     r ON f.region      = r.region
GROUP BY
    c.customer_name, c.segment,
    p.category, p.sub_category,
    r.region, r.state,
    f.order_year, f.order_quarter
ORDER BY total_sales DESC
LIMIT 20


## 9. Print a complete inventory of the Gold layer

In [0]:
print("=== GOLD LAYER CATALOGUE ===\n")

gold_tables = {
    FACT_TABLE   : "Central fact table — one row per order line item",
    DIM_CUSTOMER : "Customer dimension — who bought",
    DIM_PRODUCT  : "Product dimension  — what was bought",
    DIM_REGION   : "Region dimension   — where it was sold",
}

for table_name, description in gold_tables.items():
    count   = spark.table(table_name).count()
    cols    = len(spark.table(table_name).columns)
    print(f"  {table_name}")
    print(f"    Description : {description}")
    print(f"    Rows        : {count:,}")
    print(f"    Columns     : {cols}")
    print()

print("  Star schema: ✅ Complete")


## 10. Notebook completion summary

In [0]:
print("=" * 55)
print("  GOLD DIMENSION TABLES — COMPLETE")
print("=" * 55)
print(f"  dim_customer : {spark.table(DIM_CUSTOMER).count():,} customers")
print(f"  dim_product  : {spark.table(DIM_PRODUCT).count():,} products")
print(f"  dim_region   : {spark.table(DIM_REGION).count():,} geo combos")
print()
print(f"  Star schema referential integrity : ✅")
print(f"  All Gold tables queryable via SQL : ✅")
print(f"  Ready for analytics layer         : ✅")
print("=" * 55)

log("INFO", "06_gold_dim_tables", "Notebook 06 complete ✅")